# M5 — Golden set, serving e feedback

Este notebook apresenta as evidências geradas por `src.evaluation.golden_set`. A API usa a política fixa aprovada por padrão porque o Thompson Sampling não passou no gate do M4.

In [ ]:
from pathlib import Path
import json

import pandas as pd

# Localiza a raiz mesmo quando o kernel inicia dentro de notebooks/.
project_root = Path.cwd().resolve()
if not (project_root / 'configs').exists():
    project_root = project_root.parent
report_path = project_root / 'reports/serving/golden_set_results.json'
assert report_path.exists(), 'Execute python -m src.evaluation.golden_set antes.'
report = json.loads(report_path.read_text(encoding='utf-8'))
report['readiness']

## Cinco decisões revisadas

Os casos são sintéticos. A revisão humana explica limites e nunca transforma associações históricas em causalidade.

In [ ]:
golden_table = pd.DataFrame([
    {
        'case_id': case['case_id'],
        'scenario': case['scenario'],
        'action': case['recommended_action'],
        'fallback': case['used_fallback'],
        'review': case['human_review']['status'],
        'contract_passed': case['contract_passed'],
    }
    for case in report['cases']
])
golden_table

## Contrato operacional

- `POST /v1/recommendations` valida contexto, autorização e ações elegíveis.
- `POST /v1/feedback` aceita uma recompensa terminal por `recommendation_id`.
- `GET /health`, `/ready` e `/metrics` separam vida, prontidão e observabilidade.
- O SQLite guarda somente os dois campos de segmento necessários ao feedback, sem o payload completo.
- O modo `adaptive_demo` precisa ser solicitado explicitamente e não representa promoção.

In [ ]:
# Mostra um payload reproduzível sem iniciar servidor ou gravar uma decisão.
example = json.loads(
    (project_root / 'examples/recommendation_request.json').read_text(encoding='utf-8')
)
example

## Conclusão

Os cinco contratos foram aprovados, incluindo fallback quando celular está indisponível. Isso comprova estabilidade técnica, não benefício causal ou autorização para contato real.